# Parameters vs Buffers

**Parameters** are learned (`requires_grad=True`). **Buffers** are state (e.g. BatchNorm running stats) saved in `state_dict` but not updated by optimizers.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Inspect `named_parameters()`

In [ ]:
class DemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 3)
        self.register_buffer('running_mean', torch.zeros(3))

model = DemoNet()
for name, p in model.named_parameters():
    print(f"{name:20s} shape={tuple(p.shape)} requires_grad={p.requires_grad}")

## 2. `state_dict` keys — parameters + buffers

In [ ]:
sd = model.state_dict()
print("state_dict keys:", list(sd.keys()))

## 3. Visualize parameter tensor shapes

In [ ]:
names, sizes = [], []
for name, p in model.named_parameters():
    names.append(name.replace('.', '\n'))
    sizes.append(p.numel())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(names, sizes, color='steelblue', edgecolor='black')
ax.set_title('Parameter element counts'); ax.set_ylabel('# elements')
plt.tight_layout(); plt.show()

## 4. Heatmap of weight matrix

In [ ]:
W = model.fc.weight.detach()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(W.numpy(), cmap='RdBu_r', aspect='auto')
axes[0].set_title('fc.weight'); axes[0].set_xlabel('in'); axes[0].set_ylabel('out')
axes[1].bar(range(3), model.fc.bias.detach().numpy(), color='coral')
axes[1].set_title('fc.bias')
plt.tight_layout(); plt.show()

## 5. Buffers vs parameters in optimizer

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=0.01)
opt_params = [n for n, _ in model.named_parameters()]
buf_names = [n for n, _ in model.named_buffers()]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(['optimizer tracks', 'buffers only'], [len(opt_params), len(buf_names)],
       color=['seagreen', 'lightgray'], edgecolor='black')
ax.set_title('Parameters vs buffers')
plt.tight_layout(); plt.show()